# Pillow (PIL) 无损压缩
不过，通过合理配置 save() 方法的参数，你可以在无损的前提下尽可能榨取压缩空间。以下是参数的详细拆解：<br>
## 1. optimize=True (优化器)<br>
作用：这是一个布尔值开关。当设置为 True 时，Pillow 在保存图片时会尝试寻找最有效的 Encoder 策略。<br>
原理：它会指示编码器进行额外的 CPU 计算，以确定哪种 PNG Filter（过滤算法，如 Sub, Up, Average, Paeth）最适合当前的图像数据。<br>
影响：<br>
文件大小：通常能减少大约 0% - 10% 的大小（取决于原始数据的复杂程度）。<br>
性能：会显著增加保存图片时的 CPU 耗时。<br>
画质：完全无损。<br>

## 2. compress_level=9 (压缩级别)<br>
作用：控制 ZLIB 压缩算法的强度，取值范围为 0 到 9。<br>
0：不压缩，速度极快，文件极大。<br>
1：最低程度压缩（速度最快）。<br>
6：默认值，平衡了速度和大小。<br>
9：最高程度压缩（文件最小，速度最慢）。<br>
注意：compress_level 调节的是数据流的压缩率，而不是图片的像素质量。即使设为 9，图片像素依然是 100% 还原的。<br>

## 3. bits (位数控制)<br>
作用：控制输出 PNG 的位深（例如 8-bit 或 1-bit）。v
用法：img.save(output_path, "PNG", bits=8)。<br>
场景：如果你的图片本来就是黑白的或者色彩很少，强制指定较低的 bits 可以大幅减小体积。<br>

In [1]:
from PIL import Image

def compress_png_pillow(input_path, output_path):
    img = Image.open(input_path)
    # optimize=True 会进行额外的 CPU 计算以寻找最小的存储方式
    # compress_level 调节的是 ZLIB 压缩级别
    img.save(output_path, "PNG", optimize=True, compress_level=9)

#compress_png_pillow(r"pic_test_origin.png", "output_pillow1.png")

# 有损压缩
## A. 改变颜色模式（最有效）<br>
这是控制 PNG 大小的核心手段。将 RGB（24位色彩）转为 P（8位索引色）。<br>
## B. 调整图片尺寸 (Resizing)
如果分辨率过高，压缩算法再强也无济于事。
## C. 移除元数据 (Metadata)
图片中包含的 EXIF 信息（拍摄时间、GPS、设备信息）有时会占用几 KB 到几十 KB。

In [2]:
def compress_png_color(input_path, output_path):
    img = Image.open(input_path)
    # 将真彩色转换为 256 色的调色板模式
    img = img.convert("P", palette=Image.ADAPTIVE) 
    img.save(output_path, optimize=True)

#compress_png_color(r"pic_test_origin.png", "output_pillow_color.png")

In [3]:
def compress_png_size(input_path, output_path):
    img = Image.open(input_path)
    # 按比例缩小一半
    width, height = img.size
    img = img.resize((width // 2, height // 2), Image.Resampling.LANCZOS)
    img.save(output_path, optimize=True)

#compress_png_size(r"pic_test_origin.png", "output_pillow_size.png")

In [4]:
def compress_png_Metadata(input_path, output_path):
    img = Image.open(input_path)
    # 重新创建一个不带 info 的对象来保存
    data = list(img.getdata())
    new_img = Image.new(img.mode, img.size)
    new_img.putdata(data)
    new_img.save(output_path, optimize=True)

#compress_png_Metadata(r"pic_test_origin.png", "output_pillow_Metadata.png")